#Modele regresyjne
#####z wykorzystaniem sieci neuronowych

#Zadanie 1
Zaprojektuj model regresyjny do przewidywania lepkości pektyny na podstawie parametrów fizykochemicznych

##Wczytanie danych z pliku CSV i sprawdzenie struktury datasetu

In [ ]:
import pandas as pd
csv_path = '/content/pektyna.csv'
data = pd.read_csv(csv_path, sep=';')
print(data.shape)
print(data.columns)
print(data.dtypes)
data.info()
print(data.head())
print(data.head(10))
print(data.tail())

(482, 7)
Index(['stezenie[%]', 'pH', 'L', 'a', 'b', 'cond', 'lepkosc'], dtype='object')
stezenie[%]    object
pH              int64
L              object
a              object
b              object
cond           object
lepkosc        object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482 entries, 0 to 481
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   stezenie[%]  482 non-null    object
 1   pH           482 non-null    int64 
 2   L            482 non-null    object
 3   a            482 non-null    object
 4   b            482 non-null    object
 5   cond         241 non-null    object
 6   lepkosc      482 non-null    object
dtypes: int64(1), object(6)
memory usage: 26.5+ KB
  stezenie[%]  pH      L     a     b   cond lepkosc
0        1,00   3  31,69  3,09  1,39  606,5  0,5963
1        1,00   3  31,63  3,16  1,43  603,5  0,5963
2        1,00   3  31,63  3,23  1,46  603,9  0,5963
3        1,0

###Sprawdzenie kolumn z NaN

In [ ]:
data.isna().sum()

,0
stezenie[%],0
pH,0
L,0
a,0
b,0
cond,241
lepkosc,0


###Konwertowanie wybranych kolumn z formatu tekstowego na liczbowy (float) poprzez zamianę przecinków na kropki

In [ ]:
data['stezenie[%]'] = data['stezenie[%]'].str.replace(',', '.').astype(float)
data['L'] = data['L'].str.replace(',', '.').astype(float)
data['a'] = data['a'].str.replace(',', '.').astype(float)
data['b'] = data['b'].str.replace(',', '.').astype(float)
data['lepkosc'] = data['lepkosc'].str.replace(',', '.').astype(float)

##Podział zbioru

In [ ]:
X = data.drop(columns=['cond','lepkosc'])
y = data['lepkosc']

from sklearn.model_selection import train_test_split
import tensorflow as tf
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle = True
)

##Inicjalizacja modelu MLP

*Pobierz liczbę cech z danych wejściowych X*

In [ ]:
n_features = X_train.shape[1]

*Warstwa wejsciowa*

In [ ]:
warstwa_wejsciowa = tf.keras.layers.Input(shape=(n_features,))

*Warstwa ukryte*

In [ ]:
warstwa_ukryta = tf.keras.layers.Dense(32, activation='relu')(warstwa_wejsciowa)
warstwa_ukryta = tf.keras.layers.Dense(16, activation='relu')(warstwa_ukryta)

*Wartstwa wyjściowa*

In [ ]:
warstwa_wyjsciowa = tf.keras.layers.Dense(1)(warstwa_ukryta)

##*Zbudowanie modelu*

In [ ]:
model = tf.keras.Model(inputs=warstwa_wejsciowa, outputs=warstwa_wyjsciowa)

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
    loss = tf.keras.losses.Huber(delta=1.0),
    metrics=[
        "mae",
        tf.keras.metrics.MeanSquaredError(name="mse"),
        tf.keras.metrics.R2Score(name="r2"),
    ]
)

##*Proces uczenia/trenowania modelu MLP*

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 1.9251 - mae: 2.3438 - mse: 17.6135 - r2: 0.0488 - val_loss: 2.3121 - val_mae: 2.7317 - val_mse: 31.3405 - val_r2: 0.1064
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.4677 - mae: 1.8730 - mse: 12.8085 - r2: 0.3083 - val_loss: 1.9649 - val_mae: 2.3327 - val_mse: 27.0211 - val_r2: 0.2296
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.3644 - mae: 1.7753 - mse: 10.4663 - r2: 0.4348 - val_loss: 1.7882 - val_mae: 2.2126 - val_mse: 21.7538 - val_r2: 0.3797
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.2164 - mae: 1.6303 - mse: 9.1425 - r2: 0.5063 - val_loss: 1.6323 - val_mae: 2.0268 - val_mse: 19.9055 - val_r2: 0.4324
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.2053 - mae: 1.6296 - mse: 7.8546 - r2: 0.5758 - val_loss: 1.6172 - val_mae: 2.0276 - val_mse: 19.5101 - val_r2: 0.4437
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.1197 - mae: 1.5438 - mse: 7.62

##*Ewaluacja i predykcja*

In [ ]:
test_loss, test_mae, test_mse, test_r2 = model.evaluate(X_test, y_test)
print("Test loss (Huber):", test_loss)
print("Test MAE:", test_mae)
print("Test MSE:", test_mse)
print("Test R2:", test_r2)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.8107 - mae: 1.0976 - mse: 8.4187 - r2: 0.8075 
Test loss (Huber): 0.8107406497001648
Test MAE: 1.0975556373596191
Test MSE: 8.418695449829102
Test R2: 0.8074749112129211


#Zadanie 2
Skalowalny sposób dodawania dataset w tensorflow do modelu

##Podział zbioru

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values.astype('float32'),
    y.values.astype('float32'),
    test_size = 0.2,
    random_state = 42,
    shuffle = True
)
batch_size = 32

###Skalowalny sposób dodawania datasetów do modelu -

```
# tf.data.Dataset
```
Tworzymy dataset z Tensorflow z par<br>
**X_train** - macierz cech<br>
**y_train** - wektor etykiet
<br>**shuffle** - miesza dane, żeby model nie widział ich w jednej kolejności
<br>**batch** - dzieli na porcje po *batch_size* próbek
<br>**prefetch** - wczytuje kolejne batche w tle aby przyspieszyć trening


In [ ]:
train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
test_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_test, y_test))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

##Inicjalizacja modelu



In [ ]:
model = tf.keras.Model(inputs=warstwa_wejsciowa, outputs=warstwa_wyjsciowa)

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
    loss = tf.keras.losses.Huber(delta=1.0),
    metrics=[
        "mae",
        tf.keras.metrics.MeanSquaredError(name="mse"),
        tf.keras.metrics.R2Score(name="r2"),
    ]
)

## Proces uczenia/trenowania modelu

In [ ]:
history = model.fit(
    train_ds,
    epochs=100,
    validation_data=test_ds
)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.7527 - mae: 1.1510 - mse: 3.3559 - r2: 0.8344 - val_loss: 1.1070 - val_mae: 1.4694 - val_mse: 8.2846 - val_r2: 0.8105
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5163 - mae: 0.8795 - mse: 2.6133 - r2: 0.8710 - val_loss: 0.9101 - val_mae: 1.2063 - val_mse: 9.9447 - val_r2: 0.7726
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3664 - mae: 0.6288 - mse: 2.3122 - r2: 0.8859 - val_loss: 0.7994 - val_mae: 1.0123 - val_mse: 8.8647 - val_r2: 0.7973
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3216 - mae: 0.5425 - mse: 2.0667 - r2: 0.8980 - val_loss: 0.8585 - val_mae: 1.1639 - val_mse: 8.5033 - val_r2: 0.8055
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3303 - mae: 0.5768 - mse: 1.9982 - r2: 0.9014 - val_loss: 0.7176 - val_mae: 0.9419 - val_mse: 7.1091 - val_r2: 0.8374
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3052 - mae: 0.5512 - mse: 1.7278 - r2: 0.91

##*Ewaluacja i predykcja*

In [ ]:
test_loss, test_mae, test_mse, test_r2 = model.evaluate(X_test, y_test)
print("Test loss (Huber):", test_loss)
print("Test MAE:", test_mae)
print("Test MSE:", test_mse)
print("Test R2:", test_r2)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0474 - mae: 0.2201 - mse: 0.0948 - r2: 0.9978
Test loss (Huber): 0.04742472991347313
Test MAE: 0.2200910449028015
Test MSE: 0.09484945982694626
Test R2: 0.9978309273719788


#Zadanie 3
Na podstawie ostatnich ćwiczeń wykorzystaj klasę MLP do regresji dla zbioru z zadania 1. Przygotuj model i przedstaw wyniki

In [ ]:
import keras
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
class MLPmodel(keras.Model):
  def __init__(self):
    super(MLPmodel, self).__init__()
    self.warstwa1 = keras.layers.Dense(64, activation='relu')
    self.warstwa2 = keras.layers.Dense(32, activation='relu')
    # Change activation from 'softmax' to None for regression
    self.wyjscie = keras.layers.Dense(1, activation=None)

  def call(self, x):
    x = self.warstwa1(x)
    x = self.warstwa2(x)
    return self.wyjscie(x)

In [ ]:
import pandas as pd
csv_path = '/content/pektyna.csv'
data = pd.read_csv(csv_path, sep=';')

# Convert relevant columns from string to float, replacing commas with dots
data['stezenie[%]'] = data['stezenie[%]'].str.replace(',', '.').astype(float)
data['L'] = data['L'].str.replace(',', '.').astype(float)
data['a'] = data['a'].str.replace(',', '.').astype(float)
data['b'] = data['b'].str.replace(',', '.').astype(float)
data['lepkosc'] = data['lepkosc'].str.replace(',', '.').astype(float)

X = data.drop(columns=['cond','lepkosc'])
y = data['lepkosc']

from sklearn.model_selection import train_test_split
import tensorflow as tf
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle = True
)

In [ ]:
model = MLPmodel()
model.compile(
    optimizer='adam',
    # Change loss to a regression-appropriate one, e.g., 'mse' or Huber
    loss=tf.keras.losses.MeanSquaredError(),
    # Change metrics to regression-appropriate ones
    metrics=['mae', 'mse', tf.keras.metrics.R2Score(name='r2')]
)

In [ ]:
history = model.fit(
    X_train.values.astype('float32'), y_train.values.astype('float32'),
    epochs=100,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 18.2223 - mae: 3.0033 - mse: 18.2223 - r2: 0.0159 - val_loss: 34.4639 - val_mae: 3.3491 - val_mse: 34.4639 - val_r2: 0.0173
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 16.2523 - mae: 2.5259 - mse: 16.2523 - r2: 0.1223 - val_loss: 31.3366 - val_mae: 3.3006 - val_mse: 31.3366 - val_r2: 0.1065
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 14.4661 - mae: 2.5796 - mse: 14.4661 - r2: 0.2188 - val_loss: 28.5838 - val_mae: 3.1394 - val_mse: 28.5838 - val_r2: 0.1850
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 12.7702 - mae: 2.2314 - mse: 12.7702 - r2: 0.3104 - val_loss: 26.3964 - val_mae: 2.8115 - val_mse: 26.3964 - val_r2: 0.2474
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 11.2563 - mae: 2.1396 - mse: 11.2563 - r2: 0.3921 - val_loss: 23.9590 - val_mae: 2.5692 - val_mse: 23.9590 - val_r2: 0.3169
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 9.8186 - mae: 1.7601

In [ ]:
test_loss, test_mae, test_mse, test_r2 = model.evaluate(X_test.values.astype('float32'), y_test.values.astype('float32'))
print("Test loss (Huber):", test_loss)
print("Test MAE:", test_mae)
print("Test MSE:", test_mse)
print("Test R2:", test_r2)

Test R2: 0.927503764629364


#Zadanie 4
Na podstawie ostatnich ćwiczeń z biblioteką scikit-learn i obecnego rozwiązania regresji z klasą MLP przygotuj model i wyniki dla datasetu cukrzycy

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

print("Kształt X:", X.shape)
print("Kształt y:", y.shape)

Kształt X: (442, 10)
Kształt y: (442,)


In [ ]:
import tensorflow as tf
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle = True
)

In [ ]:
model = MLPmodel()
model.compile(
    tf.keras.optimizers.Adam(learning_rate=0.001),
    # Change loss to a regression-appropriate one, e.g., 'mse' or Huber
    loss=tf.keras.losses.MeanSquaredError(),
    # Change metrics to regression-appropriate ones
    metrics=['mae', 'mse', tf.keras.metrics.R2Score(name='r2')]
)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1, #10% z treningu idzie jako walidacja
    verbose=1
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - loss: 29934.6875 - mae: 154.3497 - mse: 29934.6875 - r2: -3.8983 - val_loss: 27480.5586 - val_mae: 147.4850 - val_mse: 27480.5586 - val_r2: -3.7975
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 29884.0664 - mae: 154.1891 - mse: 29884.0664 - r2: -3.8900 - val_loss: 27425.6328 - val_mae: 147.3014 - val_mse: 27425.6328 - val_r2: -3.7879
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 29812.0254 - mae: 153.9597 - mse: 29812.0254 - r2: -3.8782 - val_loss: 27344.3613 - val_mae: 147.0294 - val_mse: 27344.3613 - val_r2: -3.7737
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 29703.8730 - mae: 153.6189 - mse: 29703.8730 - r2: -3.8605 - val_loss: 27224.1660 - val_mae: 146.6273 - val_mse: 27224.1660 - val_r2: -3.7527
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 29546.3652 - mae: 153.1203 - mse: 29546.3652 - r2: -3.8348 - val_loss: 27049.3711 - val_mae: 146.0460 - val_mse: 27049.3711 - val_r2

In [ ]:
test_loss, test_mae, test_mse, test_r2 = model.evaluate(X_test, y_test)
print("Test loss (Huber):", test_loss)
print("Test MAE:", test_mae)
print("Test MSE:", test_mse)
print("Test R2:", test_r2)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 2965.5278 - mae: 43.1589 - mse: 2965.5278 - r2: 0.4403
Test loss (Huber): 2965.52783203125
Test MAE: 43.158851623535156
Test MSE: 2965.52783203125
Test R2: 0.440271258354187
